# LizyML Tutorial: SHAP Explanations

This notebook demonstrates how to use **SHAP (SHapley Additive exPlanations)** values
with LizyML for model interpretability on a binary classification task.

**What are SHAP values?**  
SHAP values quantify each feature's contribution to individual predictions, grounded in
cooperative game theory. Unlike split or gain importance (which are global and aggregated),
SHAP provides *per-sample* explanations that are consistent and locally accurate.

**Why they matter:**
- Identify which features drive each individual prediction
- Detect unexpected feature interactions
- Compare global importance across three methods (split, gain, SHAP)
- Debug model behaviour on specific samples

> **Requires**: `pip install 'lizyml[explain]'`

**Contents:**
1. Setup
2. Data
3. Config
4. Model Fit
5. SHAP via Predict
6. SHAP Importance Plot
7. Compare Split vs Gain vs SHAP

## 1. Setup

In [ ]:
from __future__ import annotations

import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

from lizyml import Model

## 2. Data

Synthetic binary classification dataset with 1,000 samples and 10 features.
We split off a small test set to demonstrate per-sample SHAP explanations.

In [ ]:
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    random_state=42,
)

feature_names = [f"feature_{i:02d}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

# Hold out a test set for SHAP prediction demo
df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["target"]
)
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

X_test = df_test.drop(columns=["target"])

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")
print(f"Class balance (train): {df_train['target'].mean():.1%} positive")
df_train.head()

## 3. Config

Enable SHAP by setting `explain.enabled = true`.
LizyML computes SHAP TreeExplainer values across all OOF folds during `fit()`
so that `importance(kind="shap")` and `importance_plot(kind="shap")` are available
immediately after fitting — no extra call required.

For per-sample SHAP on new data, pass `return_shap=True` to `model.predict()`.

In [ ]:
config = {
    "config_version": 1,
    "task": "binary",
    "data": {
        "target": "target",
    },
    "model": {
        "name": "lgbm",
        "params": {
            "objective": "binary",
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 4,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 5,
        },
    },
    "split": {
        "method": "stratified_kfold",
        "n_splits": 5,
    },
    "training": {
        "seed": 42,
    },
    "evaluation": {
        "metrics": ["logloss", "auc", "auc_pr", "brier"],
    },
    "explain": {
        "enabled": True,
    },
}

## 4. Model Fit

SHAP TreeExplainer values are computed per fold during cross-validation
and assembled into OOF SHAP arrays automatically.

In [ ]:
model = Model(config)
model.fit(data=df_train)
print("Fit complete.")

In [ ]:
# Quick sanity check — metrics table
model.evaluate_table().round(4)

## 5. SHAP via Predict

`model.predict(X, return_shap=True)` returns a `PredictionResult` with
a `shap_values` attribute — a 2-D array of shape `(n_samples, n_features)`
representing each feature's SHAP contribution to the log-odds of the prediction.

In [ ]:
pred = model.predict(X_test, return_shap=True)

print(f"Predictions shape:   {pred.predictions.shape}")
print(f"SHAP values shape:   {pred.shap_values.shape}")
print(f"SHAP values dtype:   {pred.shap_values.dtype}")

In [ ]:
# Per-sample SHAP values as a DataFrame
shap_df = pd.DataFrame(pred.shap_values, columns=feature_names)

print("SHAP values for first 5 test samples:")
shap_df.head().round(4)

In [ ]:
# Mean absolute SHAP per feature (global importance from test set)
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)
print("Mean |SHAP| on test set:")
mean_abs_shap.to_frame(name="mean_abs_shap").round(4)

## 6. SHAP Importance Plot

`importance_plot(kind="shap")` shows mean absolute SHAP values computed
over the **OOF predictions** from training — not the test set above.
This is the most reliable global importance estimate as it covers all training rows.

In [ ]:
model.importance_plot(kind="shap").show()

In [ ]:
# Raw OOF SHAP importance values as a Series
shap_importance = pd.Series(
    model.importance(kind="shap"), name="mean_abs_shap_oof"
).sort_values(ascending=False)
shap_importance.to_frame().round(4)

## 7. Compare Split vs Gain vs SHAP

The three importance methods tell different stories:

| Method | What it measures | Bias |
|--------|-----------------|------|
| **split** | Number of times feature used in a split | Favours high-cardinality features |
| **gain** | Average information gain per split | Favours features used in deep trees |
| **shap** | Mean absolute contribution to predictions | Consistent, locally accurate |

SHAP is generally the most trustworthy for model explanation.

In [ ]:
split_imp = pd.Series(model.importance(kind="split"), name="split")
gain_imp = pd.Series(model.importance(kind="gain"), name="gain")
shap_imp = pd.Series(model.importance(kind="shap"), name="shap")

# Rank each method (1 = most important)
comparison = pd.DataFrame(
    {
        "split_rank": split_imp.rank(ascending=False).astype(int),
        "gain_rank": gain_imp.rank(ascending=False).astype(int),
        "shap_rank": shap_imp.rank(ascending=False).astype(int),
    }
)
comparison["rank_variance"] = comparison.var(axis=1).round(2)
comparison.sort_values("shap_rank")

In [ ]:
# Side-by-side bar plots
model.importance_plot(kind="split").show()
model.importance_plot(kind="gain").show()
model.importance_plot(kind="shap").show()